<a href="https://colab.research.google.com/github/VoViet266/test/blob/master/caitienhethong.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cập nhật danh sách mirror và cài đặt poppler-utils
!apt-get update -y
!apt-get install -y --fix-missing poppler-utils
# Cài đặt các thư viện python, bổ sung bitsandbytes để chạy 4-bit
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q "Pillow>=10.4.0,<11.0.0" transformers accelerate qwen_vl_utils decord pdf2image python-docx pdfplumber pydantic

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.7 MB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,154 kB]
Fetched 13.8 MB in 3s (4,276 kB/s)
Reading package lists... Done

In [2]:
import torch
try:
    import bitsandbytes
    import transformers
    print(f"Thư viện bitsandbytes phiên bản: {bitsandbytes.__version__}")
    print(f"CUDA khả dụng: {torch.cuda.is_available()}")
    print("--- HỆ THỐNG ĐÃ SẴN SÀNG ---")
except ImportError:
    print("Vẫn chưa nhận diện được bitsandbytes. Vui lòng chọn Runtime -> Restart session.")

Thư viện bitsandbytes phiên bản: 0.50.0
CUDA khả dụng: True
--- HỆ THỐNG ĐÃ SẴN SÀNG ---


In [4]:
import os
import json
import torch
import gc
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

# Cấu hình mô hình chuẩn Qwen2-VL nhẹ
model_name = "Qwen/Qwen2-VL-2B-Instruct"
device = "cuda"

print(f"--- Đang nạp mô hình {model_name} ---")

# Dọn dẹp bộ nhớ triệt để trước khi nạp mới
if 'model' in globals(): del model
if 'processor' in globals(): del processor
torch.cuda.empty_cache()
gc.collect()

try:
    # Sử dụng float16 để tiết kiệm VRAM
    processor = AutoProcessor.from_pretrained(model_name)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    print(f"--- [OK] Hệ thống đã nạp xong Processor và Model {model_name}. ---")
except Exception as e:
    print(f"--- [LỖI] Không thể nạp mô hình: {e} ---")

--- Đang nạp mô hình Qwen/Qwen2-VL-2B-Instruct ---


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

--- [OK] Hệ thống đã nạp xong Processor và Model Qwen/Qwen2-VL-2B-Instruct. ---


In [5]:
from qwen_vl_utils import process_vision_info

def run_qwen_inference(messages, is_image=False):
    """Hàm dự đoán với max_new_tokens cao hơn để tránh mất dữ liệu"""
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    if is_image:
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(device)
    else:
        inputs = processor(text=[text], padding=True, return_tensors="pt").to(device)

    try:
        with torch.no_grad():
            # Tăng lên 2048 để chứa toàn bộ danh sách KPI
            generated_ids = model.generate(**inputs, max_new_tokens=2048)
            generated_ids_trimmed = [out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
            output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    finally:
        del inputs
        torch.cuda.empty_cache()
        gc.collect()

    clean_text = output_text.replace('```json', '').replace('```', '').strip()
    return clean_text

In [11]:
import os
import json
import re
from pdf2image import convert_from_path
from pydantic import BaseModel
from PIL import Image

class KpiRecord(BaseModel):
    ma_chi_tieu: str
    quy_danh_gia: str
    noi_dung_muc_tieu: str
    dinh_ky_thu_thap: str
    muc_dang_ky: str
    muc_dat: str
    ket_qua_he_thong: str
    nguyen_nhan: str
    hanh_dong_khac_phuc: str

def process_pdf_vision_to_kpi(path):
    if 'model' not in globals() or 'processor' not in globals():
        return "LỖI: Mô hình chưa được nạp."

    print(f"--- Đang chuyển đổi PDF sang hình ảnh: {os.path.basename(path)} ---")
    try:
        images = convert_from_path(path, dpi=200)
    except Exception as e:
        return f"Lỗi khi đọc PDF: {e}"

    all_kpis = []

    for idx, img in enumerate(images):
        print(f"--- Đang phân tích thị giác trang {idx+1}/{len(images)}... ---")
        img.thumbnail((1200, 1200))

        prompt = """Hãy tìm bảng 'Tổng hợp đánh giá mục tiêu chất lượng' và trích xuất TOÀN BỘ các dòng dữ liệu thành một mảng JSON.
Cấu trúc mỗi đối tượng JSON:
{
  "ma_chi_tieu": "Mã",
  "quy_danh_gia": "Quý 2/2026",
  "noi_dung_muc_tieu": "Nội dung",
  "dinh_ky_thu_thap": "Định kỳ",
  "muc_dang_ky": "Đăng ký",
  "muc_dat": "Thực đạt",
  "ket_qua_he_thong": "Đạt/Không đạt",
  "nguyen_nhan": "Lý do",
  "hanh_dong_khac_phuc": "Hành động"
}
Lưu ý quan trọng: Chỉ trả về mảng JSON [], không giải thích gì thêm."""

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        try:
            raw_res = run_qwen_inference(messages, is_image=True)
            # Làm sạch chuỗi: Loại bỏ Markdown code blocks nếu có
            clean_res = raw_res.strip()
            if clean_res.startswith("```"):
                clean_res = re.sub(r'^```(?:json)?|```$', '', clean_res, flags=re.MULTILINE).strip()

            # Tìm kiếm mảng JSON
            json_match = re.search(r'\[\s*\{.*\}\s*\]', clean_res, re.DOTALL)
            if json_match:
                page_data = json.loads(json_match.group())
                for item in page_data:
                    # Kiểm tra tối thiểu phải có nội dung mục tiêu
                    if item.get("noi_dung_muc_tieu") or item.get("ma_chi_tieu"):
                        all_kpis.append(item)
            else:
                print(f"Cảnh báo: Không tìm thấy định dạng JSON ở trang {idx+1}")
        except Exception as e:
            print(f"Bỏ qua trang {idx+1} do lỗi: {e}")
            continue

    return all_kpis

file_test = "/content/BM08.09.L-Tong hop danh gia MT CL&ATTT CUSC_quy 2 nam 2026 sign.pdf"
if os.path.exists(file_test):
    ket_qua_final = process_pdf_vision_to_kpi(file_test)
    print(f"\n--- HOÀN TẤT ---\nTìm thấy {len(ket_qua_final)} mục KPI.")
    print(json.dumps(ket_qua_final, indent=4, ensure_ascii=False))
    # Cập nhật biến toàn cục để lưu file
    globals()['ket_qua_final'] = ket_qua_final
else:
    print("Không tìm thấy file PDF.")

--- Đang chuyển đổi PDF sang hình ảnh: BM08.09.L-Tong hop danh gia MT CL&ATTT CUSC_quy 2 nam 2026 sign.pdf ---
--- Đang phân tích thị giác trang 1/4... ---
--- Đang phân tích thị giác trang 2/4... ---
Cảnh báo: Không tìm thấy định dạng JSON ở trang 2
--- Đang phân tích thị giác trang 3/4... ---
--- Đang phân tích thị giác trang 4/4... ---
Cảnh báo: Không tìm thấy định dạng JSON ở trang 4

--- HOÀN TẤT ---
Tìm thấy 20 mục KPI.
[
    {
        "ma_chi_tieu": "QTCL-MT01",
        "quy_danh_gia": "Quý 2/2026",
        "noi_dung_muc_tieu": "Đảm bảo hoạt động Dành giá nội bộ dựa trên quy định của ISO 9001:2015 và ISO 27001:2022",
        "dinh_ky_thu_thap": "6 tháng",
        "muc_dang_ky": "100%",
        "muc_dat": "Chưa đạt",
        "ket_qua_he_thong": "Chưa đạt",
        "nguyen_nhan": "Không có lý do",
        "hanh_dong_khac_phuc": "Không có hành động"
    },
    {
        "ma_chi_tieu": "QTCL-MT02",
        "quy_danh_gia": "Quý 2/2026",
        "noi_dung_muc_tieu": "VC-NLD được đào t

In [13]:
# Lưu kết quả trích xuất vào tệp JSON
output_file = "kpi_extracted_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(ket_qua_final, f, indent=4, ensure_ascii=False)

print(f"--- Đã lưu dữ liệu vào: {output_file} ---")
print("Bạn có thể tải tệp này từ thanh công cụ bên trái của Colab.")

--- Đã lưu dữ liệu vào: kpi_extracted_results.json ---
Bạn có thể tải tệp này từ thanh công cụ bên trái của Colab.
